# LinAge2

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import subprocess
import hashlib
import zipfile
import json
import math
from pathlib import Path

import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.LinAge2)

class LinAge2(pyagingModel):
    """Principal-component clinical clock trained on NHANES IV mortality (Fong et al. 2025)."""

    def __init__(self):
        super().__init__()
        # The 59 names the loadings, medians and MADs are indexed by, in SVD row order.
        # Set from the constants clocks/notebooks/linage2.ipynb derives when the clock is built.
        self.model_features = None
        for sex in ["male", "female"]:
            for name in ["median", "mad", "loadings", "beta", "means", "beta_null", "mean_null", "mrdt"]:
                self.register_buffer(f"{name}_{sex}", torch.empty(0))
            self.register_buffer(f"pc_index_{sex}", torch.empty(0, dtype=torch.long))
        self.register_buffer("log_mask", torch.empty(0, dtype=torch.bool))
        self.register_buffer("skip_mask", torch.empty(0, dtype=torch.bool))

    def preprocess(self, x):
        return x

    def _model_vector(self, x):
        """Assemble the 59-feature vector from the user-facing inputs.


In [3]:
model = pya.models.LinAge2()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "linage2"
model.metadata["data_type"] = "clinical biomarkers"  # Paper: we refined the clinical parameters by reducing the total number to 60
model.metadata["species"] = "Homo sapiens"  # Paper: Not all humans age at the same rate since genetics, lifestyle, and stochastic factors significantly affect future mortality and morbidity trajectories.
model.metadata["year"] = 2025
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Fong, Sheng, et al. \"LinAge2: providing actionable insights and benchmarking with epigenetic clocks.\" npj Aging 11.1 (2025): 29."
model.metadata["doi"] = "https://doi.org/10.1038/s41514-025-00221-4"
model.metadata["notes"] = "Principal-component clinical clock trained on 20-year mortality in the NHANES IV 1999-2000 wave and tested in the 2001-2002 wave. The 59 model features are log transformed where the reference specifies, robustly z-scored by sex against a healthy 40-50 year old NHANES reference, and folded at 6 MAD-scaled units; the folded vector is projected onto sex-specific singular vectors and scored by a sex-specific Cox model, so male and female samples run through entirely separate parameter sets. Chronological age is supplied in years and enters the Cox terms in months, where it is a genuine covariate rather than a cancelling offset. Sex is coded female = 1 and male = 0; a dataset with no female column scores every sample with the male parameters. C-reactive protein is supplied raw in mg/dL and takes a plain natural log with no floor, so a reading below detection coded as 0 folds to the -6 cap instead of being clamped the way the BioAge clocks clamp it. The input contract is wider than the 59 model features: total cholesterol, HDL cholesterol and triglycerides are consumed only by the Friedewald LDL and are not features themselves, and 26 questionnaire codes feed the comorbidity, self-reported-health and healthcare-use indices. An absent questionnaire block is the main hazard and it biases the estimate downward. The substituted values are the reference cohort's median profile everywhere except the comorbidity index: answering no to all 22 conditions gives 0 where the cohort's median is 1/22, so that one feature is substituted marginally healthier than the median. It costs almost nothing, because the index's entire 0 to 1 range moves the estimate by only 0.06 years. The self-reported-health index is what does the damage: its substitute of good, unchanged health is exactly the cohort median, and a subject who would have reported poor and worsening health reads about 5.3 years younger than they should. The healthcare-use index pushes the other way, by about 0.5 years for a subject who made 16 or more visits. An absent lipid panel substitutes a total cholesterol chosen so that the derived LDL lands on the reference median, avoiding the reference implementation's hard 0 mmol/L substitution; a NaN inside a lipid column that is present still takes that hard 0 path and lowers the estimate by roughly 0.35 years. The range check cannot see this at all: it resolves ranges over the clock's declared input features, and the comorbidity, self-reported-health and healthcare-use indices are derived inside the model rather than supplied, so no range check ever applies to them. Heed the missing-feature warning the prediction pipeline emits; for this clock it is the only signal that the questionnaire block was absent."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["blood", "urine"]  # Paper: albuVals <- dataMat[,"URXUMASI"] crAlbRat <- albuVals/(creaVals*1.1312*10^-4)
model.metadata["predicts"] = ["biological age"]  # Paper: computational tools that estimate individual true BA based on demographic, clinical, and/or molecular data
model.metadata["training_target"] = ["mortality"]  # Paper: clocks trained on survival and functional aging outperform those trained on chronological age
model.metadata["unit"] = ["years"]  # Paper: had BA deltas of at most 35 years with estimated BAs that never exceeded 105 years
model.metadata["model_type"] = "PCA + Cox regression"  # Paper: The loadings for male and female PCs are provided in Supplementary Table, and sex-specific weights of the Cox proportional hazards models are listed in Supplementary Table.
model.metadata["platform"] = ["clinical laboratory assays"]  # Paper: further refining the clinical parameters, especially removing serum fibrinogen due to the need for a specialized sodium citrate tube
model.metadata["population"] = "adults"  # Paper: we excluded participants top-coded at age 85 years, as we could not ascertain the exact CAs of these adults
model.metadata["journal"] = "npj Aging"
model.metadata["last_author"] = "Jan Gruber"
model.metadata["n_features"] = 85
model.metadata["citations"] = 6
model.metadata["citations_date"] = "2026-08-20"

## Download clock dependencies

In [5]:
# The paper's supplementary archive is the complete source: the reference implementation, the
# saved singular-vector matrices, the codebook, the two published example subjects and the
# NHANES table the pipeline refits on. None of it is vendored, so it is fetched here and the
# Clear directory step removes it again.
ARCHIVE_URL = (
    "https://media.springernature.com/original/springer-static/esm/"
    "art%3A10.1038%2Fs41514-025-00221-4/MediaObjects/41514_2025_221_MOESM1_ESM.zip"
)
ARCHIVE_SHA256 = "e986cdfc583f78753a2a7c759e3452ce6e87c54d1844bc7f2be3021b3e3b642e"

subprocess.run(["curl", "-sSL", "-o", "linAge2_code.zip", ARCHIVE_URL], check=True)
digest = hashlib.sha256(Path("linAge2_code.zip").read_bytes()).hexdigest()
assert digest == ARCHIVE_SHA256, f"archive contents changed: {digest}"

with zipfile.ZipFile("linAge2_code.zip") as archive:
    archive.extractall(".")

SOURCE = Path("linAge2_code")
sorted(path.name for path in SOURCE.iterdir())

['.DS_Store',
 'codebook_linAge2.csv',
 'diagDat99_F_pre.csv',
 'diagDat99_M_pre.csv',
 'linAge2.R',
 'logNoLog.csv',
 'mergedDataNHANES9902.csv',
 'paraInit.csv',
 'uMatDat99_F_pre.csv',
 'uMatDat99_M_pre.csv',
 'userData.csv',
 'userData_sanity.csv',
 'vMatDat99_F_pre.csv',
 'vMatDat99_M_pre.csv']

In [6]:
# linAge2.R ships no fitted constants: it re-runs the whole training pipeline over
# mergedDataNHANES9902.csv on every invocation and keeps the normalization statistics, the Cox
# coefficients and the null-model coefficients only in memory. So it is patched to run
# unattended and to write them out. Every replacement is anchored on an exact excerpt and
# raises rather than patching a guess.
REPLACEMENTS = [
    # The per-subject inspection loop prompts for a SEQ number and opens an X11 window. Seeding
    # its own exit sentinel makes it fall straight through: which(match(x, 0) > 0) is integer(0)
    # and both guarded branches test SEQnr > 0, so nothing runs and the loop exits.
    (
        '    SEQnr <- readline("\\n> Enter SEQ Nr. to investigate single SEQ (enter zero to exit)\\n> ")\n',
        "    SEQnr <- 0\n",
    ),
    # ggplot2 is only used by plotBars(), which that loop was the sole caller of.
    ("library(ggplot2)  ## To make figures\n", ""),
    # The archive's example data holds raw cotinine in ng/mL, which is what "C" answers.
    (
        'digiCotFlag <- readline("Have you entered cotinine values (C) or smoking status (S) ? > ")\n',
        'digiCotFlag <- "C"\n',
    ),
]

PREAMBLE_R = r"""## Prepended by clocks/notebooks/linage2.ipynb: keep any package this script needs out of the
## user's own R library, so the notebook leaves nothing behind.
local_library <- file.path(getwd(), "Rlib")
dir.create(local_library, showWarnings = FALSE)
.libPaths(c(local_library, .libPaths()))
if (!requireNamespace("survival", quietly = TRUE)) {
  install.packages("survival", repos = "https://cloud.r-project.org", lib = local_library)
}

"""

EXPORT_R = r"""
############ Appended by clocks/notebooks/linage2.ipynb ############
## The median/MAD reference is not the 40-50 year olds the paper describes: it is
## RIDAGEYR <= 50 intersected with the NHANES 1999-2000 wave, complete cases only, excluding
## accidental deaths. Re-deriving it from the script's own objects is what keeps that filter
## chain exactly right.
dir.create("consts", showWarnings = FALSE)
feat <- colnames(dataMat)[-1]
write.csv(data.frame(feature = feat, lam = as.numeric(boxCox_lam[1, match(feat, colnames(boxCox_lam))])),
          "consts/features.csv", row.names = FALSE)
seqSel <- qDataMat[,"yearsNHANES"] == "9900"
ageSel <- qDataMat[,"RIDAGEYR"] <= 50
selVec <- ageSel & seqSel
dst <- dataMat_trans[selVec,]
ss <- qDataMat[selVec,"RIAGENDR"] == 1
write.csv(data.frame(
  feature = feat,
  med_m = sapply(2:ncol(dataMat_trans), function(c) median(dst[ss,c])),
  mad_m = sapply(2:ncol(dataMat_trans), function(c) mad(dst[ss,c])),
  med_f = sapply(2:ncol(dataMat_trans), function(c) median(dst[!ss,c])),
  mad_f = sapply(2:ncol(dataMat_trans), function(c) mad(dst[!ss,c])),
  skip = feat %in% c("fs1Score","fs2Score","fs3Score","LBXCOT","LBDBANO")
), "consts/normstats.csv", row.names = FALSE)
write.csv(data.frame(name = names(coxModelM$coefficients), beta = as.numeric(coxModelM$coefficients),
                     mean = as.numeric(coxModelM$means)), "consts/coxM.csv", row.names = FALSE)
write.csv(data.frame(name = names(coxModelF$coefficients), beta = as.numeric(coxModelF$coefficients),
                     mean = as.numeric(coxModelF$means)), "consts/coxF.csv", row.names = FALSE)
write.csv(data.frame(sex = c("M","F"),
                     beta = c(nullModelM$coefficients[1], nullModelF$coefficients[1]),
                     mean = c(nullModelM$means[1], nullModelF$means[1])),
          "consts/coxnull.csv", row.names = FALSE)
cat("> Wrote consts/\n")
"""


def patch_reference_script(text):
    """Make linAge2.R run unattended and dump the constants it fits in memory."""
    for old, new in REPLACEMENTS:
        if text.count(old) != 1:
            raise ValueError(f"expected exactly one occurrence of {old[:60]!r} in linAge2.R")
        text = text.replace(old, new)
    return PREAMBLE_R + text + EXPORT_R


# Never run it inside the unpacked archive: it writes scree_M.pdf, scree_F.pdf and
# userData_out.csv into the working directory.
WORK = Path("linAge2_work")
WORK.mkdir(exist_ok=True)
for csv in SOURCE.glob("*.csv"):
    shutil.copy(csv, WORK)
(WORK / "linAge2_export.R").write_text(patch_reference_script((SOURCE / "linAge2.R").read_text()))

subprocess.run(["Rscript", "linAge2_export.R"], cwd=WORK, check=True)
CONSTS = WORK / "consts"
sorted(path.name for path in CONSTS.iterdir())

Warning message:
package ‘survival’ was built under R version 4.3.2 



I) Reading data and configuration files
#######################################
> Reading parameter file: [ paraInit.csv ] ... Done
> Reading parameters ... 
   errLevl: 0.1 
   NAcut: 0.09 
   Age limits: [ 40 , 84 ] 
   Use Derived Features: 1  ... Done

> Reading NHANES training data file: [ mergedDataNHANES9902.csv ] ... 

Done
> Reading codebook file: [ codebook_linAge2.csv ] ... Done
> Reading user data file: [ userData.csv ]... Done

> Digitizing cotinine data ... Done

> Digitizing cotinine data ... Done

> Digitizing cotinine data ... Done


II) Selecting and cleaning data
###############################
> Splitting data matrix ...  selecting data ...  

 selecting qData ... 

Done
> Populating derived features ...  fs scores ... LDLV ...

...

Done
> Projecting data into PC coordinates  ... 

Done
> Projecting data into PC coordinates  ... Done
> Projecting data into PC coordinates  ... Done
Done
> Merging PC data for both training and testing data ... Done
> Calculating scree plot  ... males   determining PCA cutoff ... males  Done
> Writing out scree plot: [ scree_M.pdf ] ... Done
> Calculating scree plot  ... females   determining PCA cutoff ... females  Done
> Writing out scree plot: [ scree_F.pdf ] ... Done
> Reducing dimensionality by dropping dimensions (PCs) explaining less than 0.5 % of variance. 
> Dropped PCs beyond PC Nr. 42  ... Done

V) Building clock based on 99/00 wave
#####################################
> Defining models ... Done
> Fitting final models ... Females ... 

Males ... Done

VI) Populating BioAges for male / female SEQs
###############################################
> Calculating BioAges for test data based on LinAge2 ...Females ... Males ... Done
> Calculating BioAges for training data based on LinAge2 ... Females ... Males ... Done
> Calculating BioAges for user data based on LinAge2 ... Females ... Males ... Done
> Sort BA into testing matrix ... for both sexesDone
> Sort BA into training matrix ... for both sexesDone
> Sort BA into user matrix ... for both sexes ... Done
> Adding PCs and LinAge2 data to user data matrix ... 

 > Sanity check passed 
Done
#################################################################################
> Data for SEQs:

  8881 9106

  added to the data matrix
> Writing updated user data matrix ... <userData_out.csv>
Done
<<< 

> Wrote consts/


['coxF.csv', 'coxM.csv', 'coxnull.csv', 'features.csv', 'normstats.csv']

In [7]:
# LinAge2 speaks NHANES variable codes; pyaging speaks descriptive snake_case, one name per
# measurement package-wide. This is the full translation, and every unit was checked against the
# archive's codebook_linAge2.csv.
NHANES_TO_PYAGING = {
    # --- examination ---
    "BPXPLS": "pulse",  # 60 sec pulse, bpm
    "BPXSAR": "systolic_blood_pressure",  # mmHg
    "BPXDAR": "diastolic_blood_pressure",  # mmHg
    "BMXBMI": "body_mass_index",  # kg/m^2
    # --- urine ---
    "URXUMASI": "urine_albumin",  # mg/L
    "URXUCRSI": "urine_creatinine",  # umol/L
    # --- iron studies ---
    "LBDIRNSI": "iron",  # umol/L
    "LBDTIBSI": "total_iron_binding_capacity",  # umol/L
    "LBXPCT": "transferrin_saturation",  # %
    "LBDFERSI": "ferritin",  # ug/L
    # --- vitamins ---
    "LBDFOLSI": "folate",  # nmol/L
    "LBDB12SI": "vitamin_b12",  # pmol/L
    # --- smoking ---
    "LBXCOT": "cotinine",  # ng/mL; digitized to `smoking_intensity` before the model sees it
    # --- complete blood count ---
    "LBXWBCSI": "white_blood_cell_count",  # 10^3 cells/uL
    "LBXLYPCT": "lymphocyte_percent",  # %
    "LBXMOPCT": "monocyte_percent",  # %
    "LBXNEPCT": "neutrophil_percent",  # %
    "LBXEOPCT": "eosinophil_percent",  # %
    "LBXBAPCT": "basophil_percent",  # %
    "LBDLYMNO": "lymphocyte_count",  # 10^3 cells/uL
    "LBDMONO": "monocyte_count",  # 10^3 cells/uL
    "LBDNENO": "neutrophil_count",  # 10^3 cells/uL
    "LBDEONO": "eosinophil_count",  # 10^3 cells/uL
    "LBDBANO": "basophil_count",  # 10^3 cells/uL
    "LBXRBCSI": "red_blood_cell_count",  # 10^6 cells/uL
    "LBXHGB": "hemoglobin",  # g/dL
    "LBXHCT": "hematocrit",  # %
    "LBXMCVSI": "mean_cell_volume",  # fL
    "LBXMCHSI": "mean_cell_hemoglobin",  # pg
    "LBXMC": "mean_cell_hemoglobin_concentration",  # g/dL
    "LBXRDW": "red_cell_distribution_width",  # %
    "LBXPLTSI": "platelet_count",  # 10^3 cells/uL
    "LBXMPSI": "mean_platelet_volume",  # fL
    # --- inflammation, glycemia, cardiac ---
    "LBXCRP": "c_reactive_protein",  # mg/dL
    "LBXGH": "hemoglobin_a1c",  # %
    "SSBNP": "nt_probnp",  # pg/mL
    # --- biochemistry panel ---
    "LBDSALSI": "albumin",  # g/L
    "LBXSATSI": "alanine_aminotransferase",  # U/L
    "LBXSASSI": "aspartate_aminotransferase",  # U/L
    "LBXSAPSI": "alkaline_phosphatase",  # IU/L, numerically U/L
    "LBDSBUSI": "blood_urea_nitrogen",  # mmol/L
    "LBDSCASI": "calcium",  # mmol/L
    "LBXSC3SI": "bicarbonate",  # mmol/L
    "LBDSGLSI": "glucose",  # mmol/L
    "LBXSLDSI": "lactate_dehydrogenase",  # U/L
    "LBDSPHSI": "phosphorus",  # mmol/L
    "LBDSTBSI": "total_bilirubin",  # umol/L
    "LBDSTPSI": "total_protein",  # g/L
    "LBDSUASI": "uric_acid",  # umol/L
    "LBDSCRSI": "creatinine",  # umol/L
    "LBXSNASI": "sodium",  # mmol/L
    "LBXSKSI": "potassium",  # mmol/L
    "LBXSCLSI": "chloride",  # mmol/L
    "LBDSGBSI": "globulin",  # g/L
    # --- lipids: consumed by the Friedewald LDL, then dropped; not model features ---
    "LBDTCSI": "total_cholesterol",  # mmol/L
    "LBDHDLSI": "hdl_cholesterol",  # mmol/L
    "LBDSTRSI": "triglycerides",  # mmol/L
    # --- questionnaire, all NHANES 1999-2000 coded categoricals ---
    "BPQ020": "told_high_blood_pressure",
    "DIQ010": "told_diabetes",
    "HUQ010": "general_health_condition",
    "HUQ020": "health_compared_to_one_year_ago",
    "HUQ050": "healthcare_visits_past_year",
    "HUQ070": "hospital_overnight_past_year",
    "KIQ020": "told_weak_or_failing_kidneys",
    "MCQ010": "told_asthma",
    "MCQ053": "treated_for_anemia_past_3_months",
    "MCQ160A": "told_arthritis",
    "MCQ160B": "told_congestive_heart_failure",  # collected by the reference code but unused
    "MCQ160C": "told_coronary_heart_disease",
    "MCQ160D": "told_angina",
    "MCQ160E": "told_heart_attack",
    "MCQ160F": "told_stroke",
    "MCQ160G": "told_emphysema",
    "MCQ160I": "told_thyroid_disease",
    "MCQ160J": "told_overweight",
    "MCQ160K": "told_chronic_bronchitis",
    "MCQ160L": "told_liver_condition",
    "MCQ220": "told_cancer",
    "OSQ010A": "fractured_hip",
    "OSQ010B": "fractured_wrist",
    "OSQ010C": "fractured_spine",
    "OSQ060": "told_osteoporosis",
    "PFQ056": "confusion_or_memory_problems",
    # --- derived model features, named for what they measure ---
    "fs1Score": "comorbidity_index",
    "fs2Score": "self_reported_health_index",
    "fs3Score": "healthcare_use_index",
    "LDLV": "ldl_cholesterol",
    "crAlbRat": "urine_albumin_creatinine_ratio",
}

In [8]:
# LBXCOT is the one NHANES code whose model slot is not the raw measurement: the reference code
# digitizes cotinine into a four-level smoking-intensity code before the feature vector is built.
MODEL_SLOT_OVERRIDES = {"LBXCOT": "smoking_intensity"}

# The 22 yes/no items summed by fs1Score. MCQ160B is deliberately absent -- the reference code
# reads it and never uses it, and the denominator is 22.
COMORBIDITY_ITEMS = [
    "told_high_blood_pressure",
    "told_diabetes",
    "told_weak_or_failing_kidneys",
    "told_asthma",
    "treated_for_anemia_past_3_months",
    "told_arthritis",
    "told_coronary_heart_disease",
    "told_angina",
    "told_heart_attack",
    "told_stroke",
    "told_emphysema",
    "told_thyroid_disease",
    "told_overweight",
    "told_chronic_bronchitis",
    "told_liver_condition",
    "told_cancer",
    "fractured_hip",
    "fractured_wrist",
    "fractured_spine",
    "told_osteoporosis",
    "confusion_or_memory_problems",
    "hospital_overnight_past_year",
]

# `foldOutliers` (linAge2.R:331-352) loops over every column with no skip list, so the fold is not
# the second half of z-scoring -- it applies to all 59 features, the five skip columns included.
FOLD = {
    "cap": 6,
    "applies_to": "all 59 features, including the five in skip_mask",
    "note": (
        "clip(value, -6, +6) after z-scoring. The five skip_mask columns are never z-scored but are "
        "still clipped, so their raw scale meets the same cap: self_reported_health_index and "
        "healthcare_use_index both range 0-8 and are capped at 6, and log(0) reaches the vector as "
        "-inf and folds to exactly -6. Do not infer that skipping normalization means skipping the "
        "fold -- the registry bounds for those two features are their input range, not their "
        "post-fold range."
    ),
}

DERIVED = {
    "smoking_intensity": {
        "nhanes": "LBXCOT",
        "inputs": ["cotinine"],
        "recipe": "step function on cotinine ng/mL: <10 -> 0, <100 -> 1, <200 -> 2, else 3",
    },
    "comorbidity_index": {
        "nhanes": "fs1Score",
        "inputs": COMORBIDITY_ITEMS,
        "recipe": (
            "count of the 22 items answered 'yes' (code 1), divided by 22; told_diabetes also "
            "counts code 3 (borderline); missing items default to 'no'"
        ),
    },
    "self_reported_health_index": {
        "nhanes": "fs2Score",
        "inputs": ["general_health_condition", "health_compared_to_one_year_ago"],
        "recipe": (
            "((health == 4) * 2 + (health == 5) * 4) * (1 - (versus == 1) * 0.5 + (versus == 2)); "
            "missing defaults are health = 3 and versus = 3, giving 0. Skips z-scoring but NOT the "
            "fold: its 0-8 range is clipped to 6, so 'poor and worsening' health saturates at 6"
        ),
    },
    "healthcare_use_index": {
        "nhanes": "fs3Score",
        "inputs": ["healthcare_visits_past_year"],
        "recipe": (
            "the raw HUQ050 visit-count code used as a number, with 77, 99 and missing mapped to 0. "
            "Skips z-scoring but NOT the fold: codes 7 and 8 are both clipped to 6"
        ),
    },
    "ldl_cholesterol": {
        "nhanes": "LDLV",
        "inputs": ["total_cholesterol", "triglycerides", "hdl_cholesterol"],
        "recipe": (
            "Friedewald: total_cholesterol - triglycerides / 5 - hdl_cholesterol, in mmol/L. The /5 "
            "divisor is the mg/dL constant applied to mmol/L inputs; that is what the reference does. "
            "The reference substitutes 0 when any input is missing"
        ),
    },
    "urine_albumin_creatinine_ratio": {
        "nhanes": "crAlbRat",
        "inputs": ["urine_albumin", "urine_creatinine"],
        "recipe": "urine_albumin / (urine_creatinine * 1.1312e-4), mg albumin per g creatinine",
    },
}

# Inputs that are read but never reach the model vector, so they are not part of the contract.
NON_INPUT_COLUMNS = {"SEQN", "RIAGENDR", "RIDAGEEX"}

In [9]:
def model_slot(code):
    """The pyaging name of the feature-vector slot NHANES variable ``code`` fills."""
    return MODEL_SLOT_OVERRIDES.get(code, NHANES_TO_PYAGING[code])


def per_sex_constants(suffix, normstats, loadings, cox, beta_null, mean_null):
    pc_index = [int(name.removeprefix("PC")) for name in cox["name"][1:]]
    return {
        "median": normstats[f"med_{suffix}"].tolist(),
        "mad": normstats[f"mad_{suffix}"].tolist(),
        "loadings": loadings.values.tolist(),
        "pc_index": pc_index,
        "beta": cox["beta"].tolist(),
        "means": cox["mean"].tolist(),
        "beta_null": beta_null,
        "mean_null": mean_null,
        # The reference rounds the mortality rate doubling time to two decimals before scaling
        # the risk delta with it. Skipping the rounding shifts the biological age in the 4th decimal.
        "mrdt": round(math.log(2) / beta_null, 2),
    }


def validation_cases():
    """The paper's two published subjects, with their inputs renamed to pyaging's convention."""
    user_data = pd.read_csv(SOURCE / "userData.csv")
    published = {8881: 88.69, 9106: 64.36}
    cases = []
    for _, row in user_data.iterrows():
        seqn = int(row["SEQN"])
        inputs = {NHANES_TO_PYAGING[code]: float(row[code]) for code in row.index if code not in NON_INPUT_COLUMNS}
        cases.append(
            {
                "seqn": seqn,
                # RIDAGEEX is age at examination in months; pyaging's `age` is years.
                "age_years": float(row["RIDAGEEX"]) / 12,
                "female": int(row["RIAGENDR"]) == 2,
                "inputs": inputs,
                "biological_age": published[seqn],
            }
        )
    return cases


features_csv = pd.read_csv(CONSTS / "features.csv")
normstats = pd.read_csv(CONSTS / "normstats.csv")
cox_null = pd.read_csv(CONSTS / "coxnull.csv").set_index("sex")

codes = features_csv["feature"].tolist()
if normstats["feature"].tolist() != codes:
    raise ValueError("normstats.csv is not in the same feature order as features.csv")

# The codebook classifies every variable, so let it decide which inputs are questionnaire items
# rather than guessing from the code prefix.
kind = pd.read_csv(SOURCE / "codebook_linAge2.csv").set_index("Var")["Demo/Exam/Quest/Lab/Mort"]

# Codes that only ever exist as a derivation, so the user supplies their ingredients instead.
computed_codes = {spec["nhanes"] for spec in DERIVED.values()} - set(MODEL_SLOT_OVERRIDES)
numeric_inputs = [NHANES_TO_PYAGING[code] for code in codes if code not in computed_codes]
# The three lipids are consumed by the Friedewald LDL and then dropped, so they are inputs
# without being features.
numeric_inputs += [NHANES_TO_PYAGING[code] for code in ("LBDTCSI", "LBDHDLSI", "LBDSTRSI")]
questionnaire_inputs = [name for code, name in NHANES_TO_PYAGING.items() if kind.get(code) == "Q"]

params = {
    "features": [model_slot(code) for code in codes],
    "inputs": {
        "numeric": numeric_inputs,
        "questionnaire": questionnaire_inputs,
        "female": "female",
        "age": "age",
    },
    "nhanes_to_pyaging": NHANES_TO_PYAGING,
    "derived": DERIVED,
    "fold": FOLD,
    # lambda is either 0 (natural log) or NA (identity); no other Box-Cox power occurs.
    "log_mask": features_csv["lam"].eq(0).tolist(),
    "skip_mask": normstats["skip"].tolist(),
    # The loading matrices are the one constant the archive ships fitted, so read them from it.
    "male": per_sex_constants(
        "m",
        normstats,
        pd.read_csv(SOURCE / "vMatDat99_M_pre.csv"),
        pd.read_csv(CONSTS / "coxM.csv"),
        float(cox_null.loc["M", "beta"]),
        float(cox_null.loc["M", "mean"]),
    ),
    "female": per_sex_constants(
        "f",
        normstats,
        pd.read_csv(SOURCE / "vMatDat99_F_pre.csv"),
        pd.read_csv(CONSTS / "coxF.csv"),
        float(cox_null.loc["F", "beta"]),
        float(cox_null.loc["F", "mean"]),
    ),
    "validation": validation_cases(),
}

if len(numeric_inputs) != 57 or len(questionnaire_inputs) != 26:
    raise ValueError(f"expected 57 numeric and 26 questionnaire inputs, got {len(numeric_inputs)} and {len(questionnaire_inputs)}")

params["features"]

['pulse',
 'systolic_blood_pressure',
 'diastolic_blood_pressure',
 'body_mass_index',
 'urine_albumin',
 'urine_creatinine',
 'iron',
 'total_iron_binding_capacity',
 'transferrin_saturation',
 'ferritin',
 'folate',
 'vitamin_b12',
 'smoking_intensity',
 'white_blood_cell_count',
 'lymphocyte_percent',
 'monocyte_percent',
 'neutrophil_percent',
 'eosinophil_percent',
 'basophil_percent',
 'lymphocyte_count',
 'monocyte_count',
 'neutrophil_count',
 'eosinophil_count',
 'basophil_count',
 'red_blood_cell_count',
 'hemoglobin',
 'hematocrit',
 'mean_cell_volume',
 'mean_cell_hemoglobin',
 'mean_cell_hemoglobin_concentration',
 'red_cell_distribution_width',
 'platelet_count',
 'mean_platelet_volume',
 'c_reactive_protein',
 'hemoglobin_a1c',
 'nt_probnp',
 'albumin',
 'alanine_aminotransferase',
 'aspartate_aminotransferase',
 'alkaline_phosphatase',
 'blood_urea_nitrogen',
 'calcium',
 'bicarbonate',
 'glucose',
 'lactate_dehydrogenase',
 'phosphorus',
 'total_bilirubin',
 'total_pro

## Load features

In [10]:
model.features = params["inputs"]["numeric"] + params["inputs"]["questionnaire"] + ["age", "female"]
len(model.features), model.features[:5], model.features[-5:]

(85,
 ['pulse',
  'systolic_blood_pressure',
  'diastolic_blood_pressure',
  'body_mass_index',
  'urine_albumin'],
 ['fractured_spine',
  'told_osteoporosis',
  'confusion_or_memory_problems',
  'age',
  'female'])

## Load weights into base model

In [11]:
model.base_model = torch.nn.Identity()

model.model_features = params["features"]
model.log_mask = torch.tensor(params["log_mask"], dtype=torch.bool)
model.skip_mask = torch.tensor(params["skip_mask"], dtype=torch.bool)

for sex in ["male", "female"]:
    fit = params[sex]
    for key in ["median", "mad", "loadings", "beta", "means", "beta_null", "mean_null", "mrdt"]:
        setattr(model, f"{key}_{sex}", torch.tensor(fit[key], dtype=torch.float64))
    setattr(model, f"pc_index_{sex}", torch.tensor(fit["pc_index"], dtype=torch.long))
    # mrdt is round(ln(2) / beta_null, 2) in the reference. The rounding is load-bearing, so the
    # stored value is used as-is; recomputing it unrounded moves the fourth decimal of the age.
    assert fit["mrdt"] == round(math.log(2) / fit["beta_null"], 2)

# Every name the 59-vector needs is either a user-facing input or one of the six quantities
# postprocess derives, and the comorbidity item list must be the same 22 the JSON records.
assert set(model.model_features) - set(model.features) == set(params["derived"])
assert set(pya.models._models.LINAGE2_COMORBIDITY_ITEMS) == set(
    params["derived"]["comorbidity_index"]["inputs"]
)
assert len(pya.models._models.LINAGE2_COMORBIDITY_ITEMS) == 22

model.mrdt_male, model.mrdt_female

(tensor(103.9500, dtype=torch.float64), tensor(82.4300, dtype=torch.float64))

## Load reference values

In [12]:
LIPID_HDL = 1.3  # mmol/L, mid-normal adult
LIPID_TRIGLYCERIDES = 1.3  # mmol/L, mid-normal adult

reference_median = {
    name: math.exp((male + female) / 2) if logged else (male + female) / 2
    for name, male, female, logged in zip(
        params["features"], params["male"]["median"], params["female"]["median"], params["log_mask"]
    )
}

substitute = {name: 2.0 for name in params["inputs"]["questionnaire"]}  # 2 = no
substitute.update(
    {
        "general_health_condition": 3.0,  # good
        "health_compared_to_one_year_ago": 3.0,  # about the same
        "healthcare_visits_past_year": reference_median["healthcare_use_index"],
        "cotinine": 0.0,  # any value under 10 ng/mL digitizes to intensity 0
        "hdl_cholesterol": LIPID_HDL,
        "triglycerides": LIPID_TRIGLYCERIDES,
        "total_cholesterol": reference_median["ldl_cholesterol"] + LIPID_TRIGLYCERIDES / 5 + LIPID_HDL,
        "age": 62.0,  # midpoint of the 40-84 training window
        "female": 0.0,  # no sex column means the male parameters
    }
)

model.reference_values = [substitute.get(name, reference_median.get(name)) for name in model.features]

assert len(model.reference_values) == len(model.features)
assert None not in model.reference_values

# Every substituted value has to be plausible on its own, not merely neutral in aggregate.
for record, value in zip(
    pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"]), model.reference_values
):
    assert record["low"] <= value <= record["high"], (record["feature"], value)

pd.DataFrame({"feature": model.features, "reference": model.reference_values})

,feature,reference
0,pulse,71.000000
1,systolic_blood_pressure,120.000000
2,diastolic_blood_pressure,76.500000
3,body_mass_index,28.077449
4,urine_albumin,7.300000
...,...,...
80,fractured_spine,2.000000
81,told_osteoporosis,2.000000
82,confusion_or_memory_problems,2.000000
83,age,62.000000


## Load preprocess and postprocess objects

In [13]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [14]:
model.postprocess_name = "linage2"
model.postprocess_dependencies = None

## Check all clock parameters

In [15]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Fong, Sheng, et al. "LinAge2: providing actionable insights and '
             'benchmarking with epigenetic clocks." npj Aging 11.1 (2025): 29.',
 'citations': 6,
 'citations_date': '2026-08-20',
 'clock_name': 'linage2',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.1038/s41514-025-00221-4',
 'journal': 'npj Aging',
 'last_author': 'Jan Gruber',
 'model_type': 'PCA + Cox regression',
 'n_features': 85,
 'notes': 'Principal-component clinical clock trained on 20-year mortality in '
          'the NHANES IV 1999-2000 wave and tested in the 2001-2002 wave. The '
          '59 model features are log transformed where the reference '
          'specifies, robustly z-scored by sex against a healthy 40-50 year '
          'old NHANES reference, and folded at 6 MAD-scaled units; the folded '
       

## Normal feature ranges

In [16]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges)

,feature,unit,low,high
0,pulse,bpm,20.0,250.0
1,systolic_blood_pressure,mmHg,50.0,260.0
2,diastolic_blood_pressure,mmHg,20.0,200.0
3,body_mass_index,kg/m^2,8.0,100.0
4,urine_albumin,mg/L,0.1,20000.0
...,...,...,...,...
80,fractured_spine,"NHANES yes/no code (1 = yes, 2 = no, 7 = refus...",1.0,9.0
81,told_osteoporosis,"NHANES yes/no code (1 = yes, 2 = no, 7 = refus...",1.0,9.0
82,confusion_or_memory_problems,"NHANES yes/no code (1 = yes, 2 = no, 7 = refus...",1.0,9.0
83,age,years,0.0,122.5


## Basic test

In [17]:
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = torch.tensor(
    [[(record["low"] + record["high"]) / 2 for record in records]], dtype=torch.float64
)
model.eval()
model.to(torch.float64)
pred = model(midpoints)
pred

tensor([[109.6068]], dtype=torch.float64)

#### Parity with the published example subjects

In [18]:
rows = [dict(case["inputs"], age=case["age_years"], female=float(case["female"])) for case in params["validation"]]

matrix = torch.tensor(
    [[float(row[name]) for name in model.features] for row in rows], dtype=torch.float64
)
with torch.inference_mode():
    predicted = model(matrix).squeeze(-1)

expected = torch.tensor([case["biological_age"] for case in params["validation"]], dtype=torch.float64)
print("predicted:", predicted.tolist())
print("published:", expected.tolist())
print("max absolute difference:", (predicted - expected).abs().max().item())

predicted: [88.69448735215042, 64.35789927322698]
published: [88.69, 64.36]
max absolute difference: 0.004487352150420065


## Save torch model

In [19]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [20]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted folder: linAge2_work
Deleted file: linAge2_code.zip
Deleted folder: __MACOSX
Deleted folder: linAge2_code
